In [1]:
"""
Weather Fire Risk Modeling (TabFM only)
==========================================
Modular pipeline using TabFM, a zero-shot tabular foundation model from
Google Research (in-context learning -- no gradient training on your
data; see https://github.com/google-research/tabfm).
 
  train_tabfm() -> fits a TabFM classifier or regressor
 
LATITUDE, LONGITUDE, NEAREST_WEATHER_LAT, and NEAREST_WEATHER_LON are
excluded from model training (per request), as is DISCOVERYDATETIME (a
raw timestamp string that isn't usable as a numeric feature as-is).
These columns are NOT dropped from the data entirely -- their test-split
values are carried through and attached to results_df alongside the
Actual/Predicted columns, so you can still see them for reference.
 
Requires:
    pip install pandas numpy scikit-learn
    # TabFM must be installed from source (no PyPI package yet):
    #   git clone https://github.com/google-research/tabfm.git
    #   cd tabfm && pip install -e .[pytorch]
 
NOTE ON DATA LEAKAGE:
WEATHER_RISK_CATEGORY is a deterministic binning of WEATHER_FIRE_RISK_SCORE
(High = 60.01-79.99, Moderate = 40.01-60.00, Low = 21.10-40.00,
Very Low = 15.00-20.00). Each target is excluded from the other task's
input columns so a model can't just "read off" the answer.
"""

'\nWeather Fire Risk Modeling (TabFM only)\n==========================================\nModular pipeline using TabFM, a zero-shot tabular foundation model from\nGoogle Research (in-context learning -- no gradient training on your\ndata; see https://github.com/google-research/tabfm).\n\n  train_tabfm() -> fits a TabFM classifier or regressor\n\nLATITUDE, LONGITUDE, NEAREST_WEATHER_LAT, and NEAREST_WEATHER_LON are\nexcluded from model training (per request), as is DISCOVERYDATETIME (a\nraw timestamp string that isn\'t usable as a numeric feature as-is).\nThese columns are NOT dropped from the data entirely -- their test-split\nvalues are carried through and attached to results_df alongside the\nActual/Predicted columns, so you can still see them for reference.\n\nRequires:\n    pip install pandas numpy scikit-learn\n    # TabFM must be installed from source (no PyPI package yet):\n    #   git clone https://github.com/google-research/tabfm.git\n    #   cd tabfm && pip install -e .[pytorch

In [2]:
import warnings
warnings.filterwarnings("ignore")
 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)
 
try:
    from tabfm import TabFMClassifier, TabFMRegressor
    from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1_0_0_backend
    TABFM_AVAILABLE = True
except ImportError:
    TABFM_AVAILABLE = False

In [3]:
RANDOM_STATE = 42
DATA_PATH = "./0.Results_WeatherWildfireModel_PrepareDataSet/filtered_training_data_selectedColumn_Category_together.csv"
CLASSIFICATION_TARGET = "WEATHER_FIRE_RISK_CATEGORY"
REGRESSION_TARGET = "WEATHER_FIRE_RISK_SCORE"

In [4]:
# Columns to exclude from model training but keep tracked in the results
# (LATITUDE/LONGITUDE/NEAREST_WEATHER_LAT/NEAREST_WEATHER_LON per request;
# DISCOVERYDATETIME added because it's a raw timestamp string that breaks
# training as-is -- see note in the response).
# Columns to exclude from model training but keep tracked in the results
# (LATITUDE/LONGITUDE/NEAREST_WEATHER_LAT/NEAREST_WEATHER_LON per request;
# DISCOVERYDATETIME added because it's a raw timestamp string that breaks
# training as-is -- see note in the response).
EXCLUDE_COLS = [
   'LATITUDE', 'LONGITUDE', 'NEAREST_WEATHER_LAT', 'NEAREST_WEATHER_LON', 'DISCOVERYDATETIME',  
   #'avg_max_temp_7d', 'avg_min_temp_7d', 'avg_rh_7d', 'total_precip_7d', 'max_wind_7d', 'consec_dry_days_7d', 'avg_solar_7d', 
   #'max_temp_7d', 'min_rh_7d', 'max_solar_7d',
   #'avg_max_temp_30d', 'total_precip_30d', 'precip_anomaly_30d', 'avg_wind_30d', 'avg_rh_30d', 'days_no_rain_30d', 'avg_solar_30d', 
   #'avg_max_temp_90d', 'max_temp_90d', 'total_precip_90d', 'days_no_rain_90d', 'avg_rh_90d', 'total_solar_90d', 
   #'days_above_heat_threshold', 'days_above_wind_threshold', 'days_low_rh', 
   #'IGNITION_WEATHER_SCORE', 'SEASONAL_SCORE', 'ASPECT_SCORE', 'ELEVATION_SCORE', 'SLOPE_WIND_SCORE', 'SPREAD_RATE_SCORE',
   #'WEATHER_FIRE_RISK_SCORE', 
   #'WEATHER_RISK_CATEGORY'
]
# Note The commented out columns will remain in the dataframe

In [5]:
# ------------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------------
def load_data(path):
    """Reads the CSV and returns a single DataFrame."""
    df = pd.read_csv(path)
    return df

In [6]:
# ------------------------------------------------------------------
# 2. Split data
# ------------------------------------------------------------------
def split_data(df, target_col, task_type, exclude_cols=None, test_size=0.30, random_state=RANDOM_STATE):
    """Splits a dataframe into 70% train / 30% test for the given target
    column. Stratifies on the target for classification tasks.
 
    exclude_cols: columns that should NOT be used as model inputs, but
    whose test-split values are still returned (as `tracking_test`) so
    they can be reattached to the results later.
    """
    exclude_cols = [c for c in (exclude_cols or []) if c in df.columns]
 
    X_full = df.drop(columns=[target_col])  # still includes exclude_cols
    y = df[target_col]
 
    if task_type == "classification":
        try:
            X_train_full, X_test_full, y_train, y_test = train_test_split(
                X_full, y, test_size=test_size, random_state=random_state, stratify=y
            )
        except ValueError:
            print("Warning: could not stratify (a class has too few "
                  "samples). Using a plain random split.")
            X_train_full, X_test_full, y_train, y_test = train_test_split(
                X_full, y, test_size=test_size, random_state=random_state
            )
    else:
        X_train_full, X_test_full, y_train, y_test = train_test_split(
            X_full, y, test_size=test_size, random_state=random_state
        )
 
    # Keep the excluded columns' test-split values for tracking purposes
    tracking_test = X_test_full[exclude_cols].reset_index(drop=True) if exclude_cols else None
 
    # Actual model inputs -- excluded columns dropped
    X_train = X_train_full.drop(columns=exclude_cols)
    X_test = X_test_full.drop(columns=exclude_cols)
 
    return X_train, X_test, y_train, y_test, tracking_test

In [7]:
# ------------------------------------------------------------------
# 3. Model training
# ------------------------------------------------------------------
def train_tabfm(X_train, y_train, task_type):
    """Fits TabFM, a zero-shot tabular foundation model. Unlike a
    conventionally trained model, TabFM does not learn weights from your
    data via gradient descent -- .fit() just prepares encoders/scalers
    and stores the training data as "context"; predictions at inference
    time are made via in-context learning using that context. Returns
    None if the tabfm package is not installed."""
    if not TABFM_AVAILABLE:
        print("NOTE: tabfm is not installed -- skipping TabFM model. "
              "Install from source: git clone "
              "https://github.com/google-research/tabfm.git && cd tabfm "
              "&& pip install -e .[pytorch]")
        return None
 
    backend_model = tabfm_v1_0_0_backend.load()
    y_values = y_train.values if hasattr(y_train, "values") else np.asarray(y_train)
 
    if task_type == "classification":
        model = TabFMClassifier(model=backend_model)
    else:
        model = TabFMRegressor(model=backend_model)
 
    # TabFM accepts a DataFrame directly (handles mixed numerical/categorical columns)
    model.fit(X_train, y_values)
    return model  # .predict() already returns values in the original label/score space

In [8]:
# ------------------------------------------------------------------
# 4. Test model: predict on the test split, store actual vs. predicted
# ------------------------------------------------------------------
def test_model(model, X_test, y_test, model_name, tracking_df=None):
    """Runs the model on the test split and returns a DataFrame with the
    actual and predicted values side by side. If tracking_df is provided
    (e.g. LATITUDE/LONGITUDE/etc. that were excluded from training), its
    columns are attached alongside Actual/Predicted for reference."""
    y_pred = model.predict(X_test)
    results_df = pd.DataFrame({
        "Actual": y_test.reset_index(drop=True),
        "Predicted": y_pred,
    })
    results_df["Model"] = model_name
 
    if tracking_df is not None:
        results_df = pd.concat(
            [tracking_df.reset_index(drop=True), results_df], axis=1
        )
 
    return results_df

In [9]:
# ------------------------------------------------------------------
# 5. Evaluate models: compute metrics from a results DataFrame
# ------------------------------------------------------------------
def evaluate_models(results_dict, task_type):
    """Takes a dict of {model_name: results_dataframe} (as produced by
    test_model) and returns a comparison DataFrame of evaluation metrics."""
    if not results_dict:
        print("No models were trained (missing tabfm?). "
              "Install the required package and re-run.")
        return pd.DataFrame()
 
    rows = []
 
    for model_name, results_df in results_dict.items():
        y_true = results_df["Actual"]
        y_pred = results_df["Predicted"]
 
        if task_type == "classification":
            rows.append({
                "Model": model_name,
                "Accuracy (%)": accuracy_score(y_true, y_pred) * 100,
                "Precision (macro)": precision_score(y_true, y_pred, average="macro", zero_division=0),
                "Recall (macro)": recall_score(y_true, y_pred, average="macro", zero_division=0),
                "F1-score (macro)": f1_score(y_true, y_pred, average="macro", zero_division=0),
                "Precision (weighted)": precision_score(y_true, y_pred, average="weighted", zero_division=0),
                "Recall (weighted)": recall_score(y_true, y_pred, average="weighted", zero_division=0),
                "F1-score (weighted)": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            })
        else:
            mape = mean_absolute_percentage_error(y_true, y_pred)
            rows.append({
                "Model": model_name,
                "R2": r2_score(y_true, y_pred),
                "MAE": mean_absolute_error(y_true, y_pred),
                "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
                "Percentage Accuracy (100-MAPE)": (1 - mape) * 100,
            })
 
    comparison_df = pd.DataFrame(rows).set_index("Model").round(4)
    return comparison_df

In [10]:
# ------------------------------------------------------------------
# 6. Compare models: display (and optionally save) the comparison table
# ------------------------------------------------------------------
def compare_models(comparison_df, title, save_path=None):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    if comparison_df.empty:
        print("(nothing to compare)")
        return comparison_df
    print(comparison_df)
    if save_path:
        comparison_df.to_csv(save_path)
        print(f"Saved comparison table to {save_path}")
    return comparison_df

In [11]:
# --- Load data ---
df = load_data(DATA_PATH)

In [12]:
# Exclude each target from the other task's inputs (see leakage note above)
df_for_classification = df.drop(columns=[REGRESSION_TARGET])
df_for_regression = df.drop(columns=[CLASSIFICATION_TARGET])

In [13]:
# --- Split data ---
Xc_train, Xc_test, yc_train, yc_test, tracking_c_test = split_data(
    df_for_classification, CLASSIFICATION_TARGET, task_type="classification",
    exclude_cols=EXCLUDE_COLS,
)
Xr_train, Xr_test, yr_train, yr_test, tracking_r_test = split_data(
    df_for_regression, REGRESSION_TARGET, task_type="regression",
    exclude_cols=EXCLUDE_COLS,
)

In [15]:
# --- Train model: classification ---
tabfm_clf = train_tabfm(Xc_train, yc_train, task_type="classification")

OSError: [Errno 524] Unknown error 524

In [ ]:
# --- Test models: classification ---
tabfm_clf_results_df = test_model(tabfm_clf, Xc_test, yc_test, "TabFM", tracking_df=tracking_c_test) \
        if tabfm_clf is not None else None

#Save the results into a csv file
tabfm_clf_results_df.to_csv('clf_results_tabfm.csv', index=False)

#Plot the data for easy visualization
m = create_map(tabfm_clf_results_df, save_path='tabfm_predicted_vs_actual.html', zoom_start=3)

In [ ]:
clf_results = {"TabFM": tabfm_clf_results_df} if tabfm_clf_results_df is not None else {}
clf_comparison = evaluate_models(clf_results, task_type="classification")
compare_models(
    clf_comparison,
    "CLASSIFICATION MODEL COMPARISON (target: WEATHER_RISK_CATEGORY)",
    save_path="classification_model_comparison_ZeroshotModels.csv",
)

In [ ]:
# --- Train models: regression ---
tabfm_reg = train_tabfm(Xr_train, yr_train, task_type="regression")

In [ ]:
# --- Test models: regression ---
tabfm_reg_results_df = test_model(tabfm_reg, Xr_test, yr_test, "TabFM", tracking_df=tracking_r_test) \
        if tabfm_reg is not None else None

#Save the results into a csv file
tabfm_reg_results_df.to_csv('reg_results_tabfm.csv', index=False)

#Plot the data for easy visualization
m = create_map(tabfm_clf_results_df, save_path='tabfm_predicted_vs_actual.html', zoom_start=3)

In [ ]:
reg_results = {"TabFM": tabfm_reg_results_df} if tabfm_reg_results_df is not None else {}
reg_comparison = evaluate_models(reg_results, task_type="regression")
compare_models(
    reg_comparison,
    "REGRESSION MODEL COMPARISON (target: WEATHER_FIRE_RISK_SCORE)",
    save_path="regression_model_comparison_Zeroshot.csv",
)